# Projet final : les récoltes agricoles

**Équipe :** (nom de l'équipe et des 5 membres)

Une direction départementale de l'agriculture (fictive) vous confie trois ans de récoltes, de 2023 à 2025, par saison (printemps, automne), département et culture, avec la pluie tombée pendant chaque saison. Le ministère veut savoir quelles cultures pèsent le plus, où les rendements sont les meilleurs, et si la pluie explique les mauvaises années.

Ce notebook montre **votre démarche** : chaque étape dit ce que vous cherchez, le code, le résultat et ce que vous en concluez. Les graphiques doivent être visibles : avant de pousser, **Restart**, puis **Run All**.

Le tableau de bord (`app.py`) reprend ensuite vos résultats pour le décideur.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

brut = pd.read_csv("donnees/recoltes_brut.csv")

## 1. Diagnostic

Forme, types, valeurs manquantes, lignes en double, et toutes les écritures d'une même valeur dans les colonnes de texte. Notez chaque problème trouvé : vous le corrigerez à l'étape 2.

In [2]:
import matplotlib.pyplot as plt
import pandas as pd

brut = pd.read_csv("donnees/recoltes_brut.csv")
print("--- Forme du jeu de données brut ---")
print(brut.shape)
print("\n--- Informations sur les colonnes et types ---")
print(brut.info())

print("\n--- Valeurs manquantes par colonne ---")
print(brut.isnull().sum())

print("\n--- Nombre de lignes en double ---")
print(brut.duplicated().sum())

print("\n--- Écritures uniques de 'saison' ---")
print(brut['saison'].unique())

print("\n--- Écritures uniques de 'departement' ---")
print(sorted(brut['departement'].dropna().unique()))

print("\n--- Écritures uniques de 'culture' ---")
print(sorted(brut['culture'].dropna().unique()))

--- Forme du jeu de données brut ---
(408, 6)

--- Informations sur les colonnes et types ---
<class 'pandas.DataFrame'>
RangeIndex: 408 entries, 0 to 407
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   annee         408 non-null    int64
 1   saison        408 non-null    str  
 2   departement   408 non-null    str  
 3   culture       408 non-null    str  
 4   surface_ha    408 non-null    str  
 5   production_t  393 non-null    str  
dtypes: int64(1), str(5)
memory usage: 30.8 KB
None

--- Valeurs manquantes par colonne ---
annee            0
saison           0
departement      0
culture          0
surface_ha       0
production_t    15
dtype: int64

--- Nombre de lignes en double ---
18

--- Écritures uniques de 'saison' ---
<ArrowStringArray>
['Printemps', 'Automne']
Length: 2, dtype: str

--- Écritures uniques de 'departement' ---
[' Artibonite', ' Centre', " Grand'Anse", ' Nord', ' Nord-Est', ' Nord-Ouest',

## 2. Nettoyage

Une copie `propre`, puis une cellule par problème, en comptant les lignes avant et après chaque suppression. Dites ce que vous supprimez et pourquoi.

**Contrôle :** à la fin, `propre.shape` vaut `(376, 6)`, sans valeur manquante ni doublon, avec 2 valeurs dans `saison`, 10 valeurs dans `departement`, 7 valeurs dans `culture`.

In [4]:
cultures_ref = pd.read_csv("donnees/cultures.csv")['culture'].tolist()
dept_ref = pd.read_csv("donnees/departements.csv")['departement'].tolist()

map_dept = {d.lower(): d for d in dept_ref}
map_cult = {c.lower(): c for c in cultures_ref}

propre = brut.copy()
print(f"Lignes initiales : {len(propre)}")

propre['saison'] = propre['saison'].str.strip()
propre['departement'] = propre['departement'].str.strip().str.lower().map(map_dept)
propre['culture'] = propre['culture'].str.strip().str.lower().map(map_cult)

propre['surface_ha'] = (
    propre['surface_ha']
    .astype(str)
    .str.replace(" ", "", regex=False)
    .str.replace("ha", "", regex=False)
    .str.strip()
)
propre['surface_ha'] = pd.to_numeric(propre['surface_ha'], errors='coerce')

propre['production_t'] = (
    propre['production_t']
    .astype(str)
    .str.replace(" ", "", regex=False)
    .str.replace("t", "", regex=False)
    .str.strip()
)
propre['production_t'] = pd.to_numeric(propre['production_t'], errors='coerce')

lignes_avant_na = len(propre)
propre = propre.dropna()
print(f"Lignes supprimées pour valeurs manquantes : {lignes_avant_na - len(propre)}")

lignes_avant_dup = len(propre)
propre = propre.drop_duplicates()
print(f"Lignes supprimées pour doublons : {lignes_avant_dup - len(propre)}")

propre['annee'] = propre['annee'].astype(int)

print("\n--- CONTRÔLE NETTOYAGE ---")
print("Forme finale:", propre.shape)
print("Nombre de valeurs dans 'saison'", propre['saison'].nunique())
print("Nombre de valeurs dans 'departement' :", propre['departement'].nunique())
print("Nombre de valeurs dans 'culture' :", propre['culture'].nunique())
assert propre.shape == (376, 6), "Erreur : La forme doit être (376, 6)"

Lignes initiales : 408
Lignes supprimées pour valeurs manquantes : 15
Lignes supprimées pour doublons : 17

--- CONTRÔLE NETTOYAGE ---
Forme finale: (376, 6)
Nombre de valeurs dans 'saison' 2
Nombre de valeurs dans 'departement' : 10
Nombre de valeurs dans 'culture' : 7


## 3. Jointures

Joignez `propre` avec `cultures.csv`, `pluies.csv`, et ajoutez les colonnes calculées dont vous aurez besoin. Comptez les lignes avant et après chaque `merge`.

**Contrôle :** 376 lignes après les jointures, autant qu'avant.

In [5]:
cultures_df = pd.read_csv("donnees/cultures.csv")
pluies_df = pd.read_csv("donnees/pluies.csv")
departements_df = pd.read_csv("donnees/departements.csv")

print(f"Lignes dans 'propre' avant jointures : {len(propre)}")

complet = propre.merge(cultures_df, on="culture", how="left")
print(f"Lignes après merge cultures : {len(complet)}")

complet = complet.merge(pluies_df, on=["annee", "saison", "departement"], how="left")
print(f"Lignes après merge pluies : {len(complet)}")

complet = complet.merge(departements_df, on="departement", how="left")
print(f"Lignes après merge departements : {len(complet)}")

complet['rendement_t_ha'] = complet['production_t'] / complet['surface_ha']

complet['valeur_htg'] = complet['production_t'] * 1000 * complet['prix_htg_kg']

assert len(complet) == 376, "Erreur : Le tableau complet doit avoir 376 lignes."
print("Contrôle réussi : 376 lignes conservées après les jointures.")

Lignes dans 'propre' avant jointures : 376
Lignes après merge cultures : 376
Lignes après merge pluies : 376
Lignes après merge departements : 376
Contrôle réussi : 376 lignes conservées après les jointures.


Enregistrez le tableau complet : c'est le fichier que lit le tableau de bord.

```python
complet.to_csv("donnees/recoltes_propre.csv", index=False)
```

In [7]:
complet.to_csv("donnees/recoltes_propre.csv", index=False)
print("Fichier généré avec succès.")

Fichier généré avec succès.


## 4. Quelles cultures pèsent le plus, en tonnes puis en valeur (gourdes) ? Le classement change-t-il ?

**Ce qu'on en conclut :**

## 5. Où les rendements (tonnes par hectare) sont-ils les meilleurs, culture par culture ?

**Ce qu'on en conclut :**

## 6. La pluie explique-t-elle les rendements ? Pour quelles cultures ? Montrez-le avec des nuages de points.

**Ce qu'on en conclut :**

## 7. Quelles cultures progressent ou reculent entre 2023 et 2025 ?

**Ce qu'on en conclut :**

## 8. Rapportez la production à la population de chaque département : quels départements produisent le plus par habitant ?

**Ce qu'on en conclut :**

## 9. Conclusion

Cinq phrases au plus : ce qu'il faut retenir, ce que les données ne permettent pas de dire, ce qu'il faudrait mesurer ensuite.